# 第10章 性能評価

## 10.3 LLMを用いた自動評価

### 10.3.2 Japanese Vicuna QA Benchmarkによる自動評価

#### 環境の準備

In [ ]:
!pip install bitsandbytes 'datasets<4.0.0' transformers[torch,sentencepiece] openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.2/594.2 kB 35.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 91.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached 

In [ ]:
from transformers.trainer_utils import set_seed
set_seed(42)

#### データセットの準備

In [ ]:
from datasets import load_dataset

test_dataset = load_dataset(
    "llm-book/ja-vicuna-qa-benchmark", split="test"
)
print(test_dataset)

In [ ]:
# データを表示
test_data = test_dataset[0]
print(test_dataset[0])

#### パイプラインの作成

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

model_name = "tokyotech-llm/Swallow-7b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=False,
    device_map="auto",
)

In [ ]:
from transformers import pipeline

generation_config = {
    "do_sample": True,
    "max_new_tokens": 2048,
    "temperature": 0.99,
    "top_p": 0.95
}
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    **generation_config
)

#### 質問に回答するためのプロンプトの作成

In [ ]:
prompt_template = """
以下に、あるタスクを説明する指示があります。
リクエストを適切に完了するための回答を記述してください。\n\n### 指示:\n{instruction}\n\n### 応答:\n
"""
# プロンプトテンプレートの{instruction}に入力テキストに置換する
prompt = prompt_template.format(instruction=test_data["turns"][0])
print(prompt)